# 04 · Reproducible Build & Figure Gallery

**Stage 13 of the workflow — the notebook that proves the whole project regenerates from
the seven source CSVs with one command, and gathers every rendered figure in one place.**

The SQL files create the production tables; this notebook records the *reasoning trail* and
the *evidence*: what was built, whether it passed its own quality gates, and what each figure
shows. It reads only committed artifacts (`artifacts/*.json`, `artifacts/figures/`) and the
warehouse — so it re-runs cleanly after a rebuild.

## The one command

```bash
python src/run_all.py \
  --source-csv-dir "<folder with the 7 original CSVs>" \
  --rebuild
```

That single entry point runs, in order:

1. **`src/pipeline.py`** — loads the 7 raw CSVs → `stg` (typed views + event-derived sessions)
   → `core` (star schema) → `mart` (decision tables) → `qa` (tests) → CSV/Parquet exports.
2. **`src/advanced_analytics.py`** — Gini/Lorenz concentration analysis, RFM clustering, and
   the return-propensity model → `data/processed/advanced/` + `artifacts/`.
3. **`src/render_outputs.py`** — the 8 figures in `artifacts/figures/`.
4. **`src/build_notebooks.py`** — regenerates notebooks `01` and `03` from live warehouse
   numbers (notebook `02_statistical_deep_dives` is hand-authored and re-executed in place).
5. **`src/validate_project.py`** — 20+ contract checks (row counts, session lineage,
   metric reconciliation, PII, figure quality). A non-zero exit blocks the build.

---

## The pipeline in one flow

```mermaid
flowchart LR
    A["7 original CSVs"] --> B["raw: source-preserving load"]
    B --> C["stg: typed views"]
    C --> S["stg.sessions: events grouped by session_id"]
    C --> D["core: conformed star schema"]
    S --> D
    D --> E["mart: funnel · commercial · customer · operations · inventory"]
    E --> F["Power BI CSV / Parquet exports"]
    E --> G["Advanced Python methods"]
    D --> H["qa: tests + reconciliation"]
    G --> I["Model deliverables + figures"]
```

## 1. Connect to the executed warehouse

In [1]:
from pathlib import Path
import json
import duckdb
import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
WAREHOUSE = PROJECT_ROOT / "artifacts" / "thelook_analytics.duckdb"
FIGURES = PROJECT_ROOT / "artifacts" / "figures"

connection = duckdb.connect(str(WAREHOUSE), read_only=True)
print("Warehouse:", WAREHOUSE.name)

Warehouse: thelook_analytics.duckdb


## 2. What the build produced — table inventory by layer

In [2]:
inventory = connection.execute(
    """
    SELECT table_schema AS layer, COUNT(*) AS tables
    FROM information_schema.tables
    WHERE table_schema IN ('raw','stg','core','mart','qa')
    GROUP BY table_schema
    ORDER BY CASE table_schema
        WHEN 'raw' THEN 1 WHEN 'stg' THEN 2 WHEN 'core' THEN 3
        WHEN 'mart' THEN 4 WHEN 'qa' THEN 5 END
    """
).fetchdf()
inventory

,layer,tables
0,raw,7
1,stg,8
2,core,10
3,mart,16
4,qa,2


## 3. Did the build pass its own quality gates?

`validate_project.py` writes `artifacts/validation_report.json` and only exits 0 when every
contract holds. Here is the committed evidence, plus the live source→session lineage.

In [3]:
report = json.loads((PROJECT_ROOT / "artifacts" / "validation_report.json").read_text(encoding="utf-8"))
print("Validation status:", report["status"])
print("Errors:", len(report["errors"]), "| Warnings:", len(report["warnings"]))
print("Figures checked:", report["figures_checked"], "| Notebooks checked:", report["notebooks_checked"])
print()
print("Source → session lineage (reconciles with zero variance):")
for k, v in report["session_lineage"].items():
    print(f"  {k:28s} {v:,}")

Validation status: PASS
Errors: 0 | Warnings: 1
Figures checked: 8 | Notebooks checked: 4

Source → session lineage (reconciles with zero variance):
  source_events                2,420,661
  source_sessions              680,862
  fact_sessions                680,862
  modeled_session_events       2,420,661


### Data-quality test ledger (`qa.test_results`)

In [4]:
qa = connection.execute(
    "SELECT status, COUNT(*) AS checks FROM qa.test_results GROUP BY status ORDER BY status"
).fetchdf()
display(qa)
blocking = connection.execute("SELECT COUNT(*) FROM qa.test_results WHERE status='FAIL'").fetchone()[0]
assert blocking == 0, "Blocking data-quality failures present"
print("Blocking failures:", blocking, "→ build is releasable")

,status,checks
0,PASS,18
1,WARN,3


Blocking failures: 0 → build is releasable


## 4. Headline model results (`artifacts/advanced_metrics.json`)

The advanced layer is deliberately honest: on this synthetic dataset the return model has
almost no signal (AUC ≈ 0.50), which is the *correct* thing to report. The transferable value
is the method — a time-based holdout, pre-outcome features, and calibrated evaluation.

In [5]:
metrics = json.loads((PROJECT_ROOT / "artifacts" / "advanced_metrics.json").read_text(encoding="utf-8"))
rfm = metrics["rfm_segmentation"]
model = metrics["return_propensity"]

print(f"RFM: {rfm['customers_segmented']:,} customers -> k={rfm['selected_k']} "
      f"(silhouette {rfm['silhouette_selected_k']:.3f})")
for s in rfm["segments"]:
    print(f"    {s['segment']:18s} {s['customers']:>7,} customers  "
          f"| median recency {s['median_recency_days']:>4.0f}d  | total net ${s['total_net_sales']:,.0f}")
print(f"Return model: {model['eligible_rows']:,} eligible items | ROC-AUC {model['roc_auc']:.3f} "
      f"| test return rate {model['test_return_rate']:.1%}")

RFM: 66,215 customers -> k=3 (silhouette 0.332)
    Champions           19,550 customers  | median recency  190d  | total net $4,388,338
    New / Developing    12,977 customers  | median recency   32d  | total net $1,161,676
    Hibernating         33,688 customers  | median recency  502d  | total net $2,546,517
Return model: 63,463 eligible items | ROC-AUC 0.501 | test return rate 28.8%


## 5. Figure gallery

`src/render_outputs.py` regenerates all figures into `artifacts/figures/`. The technical check
below confirms every expected file exists; the gallery underneath curates the figures for the
three methods this project walks through hands-on (Gini, RFM, Return Propensity).

In [6]:
expected = [
    "monthly_sales_profit", "category_sales_margin", "cohort_retention_heatmap",
    "funnel_by_channel", "delivery_lead_time_dc", "rfm_segments",
    "return_model_lift", "category_association_rules",
]
present = sorted(p.stem for p in FIGURES.glob("*.png"))
missing = [f for f in expected if f not in present]
print(f"Figures present: {len(present)} / {len(expected)} expected")
assert not missing, f"Missing figures: {missing}"
present

Figures present: 8 / 8 expected


['category_association_rules',
 'category_sales_margin',
 'cohort_retention_heatmap',
 'delivery_lead_time_dc',
 'funnel_by_channel',
 'monthly_sales_profit',
 'return_model_lift',
 'rfm_segments']

### Commercial & product
![Monthly net sales and net profit](../artifacts/figures/monthly_sales_profit.png)
![Category net sales vs margin](../artifacts/figures/category_sales_margin.png)

### Customer lifecycle
![Purchase cohort retention heatmap](../artifacts/figures/cohort_retention_heatmap.png)
![RFM segments](../artifacts/figures/rfm_segments.png)

### Acquisition & operations
![Funnel by channel](../artifacts/figures/funnel_by_channel.png)
![Delivery lead time by distribution center](../artifacts/figures/delivery_lead_time_dc.png)

### Return-propensity model
![Return model decile lift](../artifacts/figures/return_model_lift.png)

In [7]:
connection.close()
print('Warehouse connection closed.')

Warehouse connection closed.


## Takeaways

- **One command, fully reproducible.** `python src/run_all.py --rebuild` rebuilds the
  warehouse, models, figures, notebooks, and validation from the seven source files.
- **The build validates itself.** 20+ contracts (row counts, session lineage, metric
  reconciliation, PII, figure quality) must pass before the run is considered releasable.
- **Honest analysis over impressive-looking analysis.** The return model reports AUC ≈ 0.50
  rather than dressing up noise; the value is the methodology, not a fake production score.
- **This notebook is the audit trail** that ties the SQL warehouse, the Python models, and
  the Power BI hand-off into a single evidence-backed story.